# EcoCool Potential Analysis: Step-by-Step Manual Execution (Emission Optimization) - Version 2

**Version 2 includes parameter calibration** - This version fits system parameters (U-value, Heat Capacity, COP) from measured data before running the optimization, providing more accurate results.

This notebook allows you to run the EcoCool potential analysis step by step with **emission-based optimization**, with each step in a separate cell. This enables you to review and verify each intermediate result.

## New in Version 2

- **Step 4.5: Parameter Calibration** - Fits thermal parameters (Ria, Ci, COP) from measured data using the calibration framework from `flexality-shared`
- Uses fitted parameters instead of estimated values for more accurate power calculations

## How to Use

1. Run cells sequentially from top to bottom
2. Review outputs after each step
3. Modify parameters in configuration cells as needed
4. Skip optional steps (e.g., PV optimization) if not needed

## Prerequisites

- All dependencies installed (`pixi install`)
- Input data files available in `data/ecocool/`:
  - `power_data_ecocool_2024.csv` - Power consumption data
  - `emission_factor_2024.csv` - CO₂ emission factors (g CO₂/kWh or kg CO₂/kWh)
  - `cleaned_fetched_data.csv` - Measured data with processIndoorTemp, processPowerEl, weatherOutdoorTemp (for calibration)
- Access to `flexality-shared/src/calibration/calibration_framework.py` for parameter fitting
- `config.py` configured with system parameters (optional)


## Step 0: Setup and Imports

**What this step does:** Imports all necessary Python libraries (pandas, numpy, matplotlib) and sets display options for better data visualization.

**What you'll see:** Confirmation message that imports were successful.


In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from datetime import datetime
from pathlib import Path
import sys

# Add flexality-shared to path for calibration framework
FLEXALITY_SHARED_PATH = Path(r"C:\Users\MichaelDotan\flexality-shared\src")
if FLEXALITY_SHARED_PATH.exists():
    sys.path.insert(0, str(FLEXALITY_SHARED_PATH))
    print(f"Added flexality-shared to path: {FLEXALITY_SHARED_PATH}")
else:
    print(f"Warning: flexality-shared path not found: {FLEXALITY_SHARED_PATH}")
    print("Parameter calibration will not be available")

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

print("Imports successful!")


## Step 0.1: Configuration

**What this step does:** Loads system parameters from `config.py` (if available) or sets default values for the EcoCool system. Defines temperature constraints, thermal properties, PCM parameters, data file paths, and analysis period.

**Note:** In Version 2, thermal properties will be fitted from measured data in Step 4.5, so initial values here are just starting points.

**What you'll see:** Configuration summary showing system name, temperature settings, COP, and analysis period.


In [ ]:
# Import configuration (optional - can override below)
try:
    from config import (
        COOLING_RAMP_SLOPE_IN_K_PER_H,
        WARMING_RAMP_SLOPE_IN_K_PER_H,
        COP,
        POWER_PRICE_IN_EURO_PER_KW,
    )
    print("Configuration loaded from config.py")
except ImportError as e:
    print(f"Warning: Could not import from config.py: {e}")
    print("Using default values...")
    # Set defaults for EcoCool
    COOLING_RAMP_SLOPE_IN_K_PER_H = -1.0
    WARMING_RAMP_SLOPE_IN_K_PER_H = 2.0
    COP = 2.5
    POWER_PRICE_IN_EURO_PER_KW = 100

# EcoCool system parameters
SYSTEM_NAME = "EcoCool"
DEFAULT_INDOOR_TEMP = -20.0  # °C (typical for deep freeze)
MIN_TEMP_ALLOWED = -25.0  # °C
MAX_TEMP_ALLOWED = -15.0  # °C

# Thermal properties (initial estimates - will be fitted in Step 4.5)
WALL_AREA = 100.0  # m² (total wall area) - needed for U-value conversion
INSULATION_THICKNESS = 0.15  # m (not used if fitting, but kept for reference)
INSULATION_TYPE = "polyurethane"  # Typical for cold storage
CONTENT_MASS = 5000.0  # kg (air + contents) - initial estimate
SPECIFIC_HEAT_CAPACITY = 1005.0  # J/(kg·K) (air) - initial estimate

# Initial parameter estimates (will be fitted)
INITIAL_RIA = 2.74  # Thermal resistance (K/kW) - initial estimate
INITIAL_CI = 3.24  # Thermal capacitance (MJ/K) - initial estimate  
INITIAL_COP = 2.5  # Coefficient of Performance - initial estimate

# PCM parameters (currently no PCM)
PCM_MASS = 0.0  # kg
LATENT_HEAT_CAPACITY = 200000.0  # J/kg (not used if PCM_MASS = 0)
PHASE_CHANGE_TEMP = -20.0  # °C (not used if PCM_MASS = 0)
LATENT_HEAT_FACTOR = 1.0  # No PCM benefit

# Data file paths - MODIFY THESE TO MATCH YOUR SYSTEM
PROJECT_ROOT = Path.cwd()
ECOCOOL_DATA_DIR = PROJECT_ROOT / "data" / "ecocool"
power_data_path = ECOCOOL_DATA_DIR / "power_data_ecocool_2024.csv"
emission_factor_path = ECOCOOL_DATA_DIR / "emission_factor_2024.csv"
measured_data_path = ECOCOOL_DATA_DIR / "cleaned_fetched_data.csv"  # For calibration

# Analysis period
start_date = pd.Timestamp('2024-05-01 00:00:00')
end_date = pd.Timestamp('2024-05-02 23:45:00')

# Calibration period (can be different from analysis period - use longer period for better fit)
calibration_start_date = pd.Timestamp('2024-05-01 00:00:00')
calibration_end_date = pd.Timestamp('2024-05-07 23:45:00')  # Use 1 week for calibration

# Report directory
report_directory = "reports/ecocool/manual_analysis_v2"
os.makedirs(report_directory, exist_ok=True)

print(f"\nConfiguration:")
print(f"  System: {SYSTEM_NAME}")
print(f"  Default temp: {DEFAULT_INDOOR_TEMP}°C")
print(f"  Temp range: {MIN_TEMP_ALLOWED}°C to {MAX_TEMP_ALLOWED}°C")
print(f"  Initial COP estimate: {INITIAL_COP}")
print(f"  Analysis period: {start_date} to {end_date}")
print(f"  Calibration period: {calibration_start_date} to {calibration_end_date}")


## Step 1: Load Power Consumption Data

**What this step does:** Loads the power consumption data from the CSV file. This is the baseline (measured) power consumption of the EcoCool cooling system.

**What you'll see:** Number of rows loaded, column names, and first few rows of raw data showing hourly power measurements.


In [ ]:
# Load power data
print(f"Loading power data from: {power_data_path}")
if not power_data_path.exists():
    raise FileNotFoundError(f"Power data file not found: {power_data_path}")

df_power = pd.read_csv(power_data_path)
print(f"\nLoaded {len(df_power)} rows")
print(f"Columns: {df_power.columns.tolist()}")
print(f"\nFirst few rows:")
print(df_power.head(30))


## Step 1.1: Process Power Data Timestamps

**What this step does:** Converts timestamp columns to datetime index, filters data for the analysis period, and resamples to 15-minute intervals. Identifies the power column and renames it to `Standortverbrauch` (site consumption).

**What you'll see:** Processed data shape, time range, power range, and first few rows with proper datetime index. Note: Hourly data will show NaN values at 15-minute intervals until forward-filled in Step 3.


In [ ]:
# Find timestamp column
timestamp_col = None
for col in ['interval', 'timestamp', 'Timestamp', 'time', 'Time', 'datetime', 'DateTime']:
    if col in df_power.columns:
        timestamp_col = col
        break

if timestamp_col is None:
    # Create timestamp index if not present
    print("No timestamp column found, creating sequential timestamps")
    df_power.index = pd.date_range(
        start='2024-01-01 00:00:00',
        periods=len(df_power),
        freq='15min'
    )
    df_power.index.name = 'timestamp'
else:
    df_power[timestamp_col] = pd.to_datetime(df_power[timestamp_col])
    df_power = df_power.set_index(timestamp_col, drop=True)

# Find power column
power_col = None
for col in ['ElectricMeter_Kaelteanlage_power', 'processPowerEl', 'power', 'Power', 'consumption', 'Consumption']:
    if col in df_power.columns:
        power_col = col
        break

if power_col is None:
    raise ValueError(f"Could not find power column. Available columns: {df_power.columns.tolist()}")

# Remove timezone if present
if df_power.index.tz is not None:
    df_power.index = df_power.index.tz_localize(None)

# Filter for analysis period
df_power = df_power[(df_power.index >= start_date) & (df_power.index <= end_date)]

# Resample to 15-minute intervals if needed
if df_power.index.freq is None or df_power.index.freq != pd.Timedelta('15min'):
    print(f"Resampling to 15-minute intervals")
    df_power = df_power.resample('15min').mean()

# Rename power column to standard name
df_power['Standortverbrauch'] = df_power[power_col]

print(f"\nAfter processing:")
print(f"  Shape: {df_power.shape}")
print(f"  Time range: {df_power.index.min()} to {df_power.index.max()}")
print(f"  Power range: {df_power['Standortverbrauch'].min():.2f} to {df_power['Standortverbrauch'].max():.2f} kW")
print(f"\nFirst few rows:")
print(df_power[['Standortverbrauch']].head(30))


## Step 2: Load Emission Factor Data

**What this step does:** Loads CO₂ emission factors from the CSV file. These factors indicate how "clean" or "dirty" the electricity grid is at each time (g CO₂/kWh).

**Understanding emission factors:**
- **Low values (clean grid):** Low CO₂ emissions per kWh - typically when renewable energy (solar, wind, hydro) is abundant
- **High values (dirty grid):** High CO₂ emissions per kWh - typically when fossil fuels (coal, natural gas) are used more

**What you'll see:** Number of rows loaded, column names, and first few rows showing emission factors over time. Higher values = dirtier grid, lower values = cleaner grid.


In [ ]:
# Load emission factors
print(f"Loading emission factors from: {emission_factor_path}")
if not emission_factor_path.exists():
    raise FileNotFoundError(f"Emission factor file not found: {emission_factor_path}")

df_emission = pd.read_csv(emission_factor_path)
print(f"\nLoaded {len(df_emission)} rows")
print(f"Columns: {df_emission.columns.tolist()}")
print(f"\nFirst few rows:")
print(df_emission.head(30))


## Step 2.1: Process Emission Factor Data

**What this step does:** Converts timestamp columns to datetime index, detects and converts units (kg to g CO₂/kWh if needed), resamples to 15-minute intervals, and removes duplicate datetime columns.

**What you'll see:** Emission factor range (min/max), time range, and first few rows. The emission factors will be used to optimize when to cool more (low emissions = clean grid) vs. less (high emissions = dirty grid).


In [ ]:
# Find timestamp column in emission data
emission_timestamp_col = None
for col in ['dateTime', 'dateTime.1', 'timestamp', 'Timestamp', 'time', 'Time', 'datetime', 'DateTime']:
    if col in df_emission.columns:
        emission_timestamp_col = col
        break

if emission_timestamp_col:
    df_emission[emission_timestamp_col] = pd.to_datetime(df_emission[emission_timestamp_col])
    df_emission = df_emission.set_index(emission_timestamp_col, drop=True)
    # Drop any other datetime columns (e.g., dateTime.1 if dateTime was used as index)
    datetime_cols_to_drop = [col for col in df_emission.columns if col in ['dateTime', 'dateTime.1', 'timestamp', 'Timestamp', 'time', 'Time', 'datetime', 'DateTime']]
    if datetime_cols_to_drop:
        df_emission = df_emission.drop(columns=datetime_cols_to_drop)

# Find emission factor column
emission_col = None
for col in ['consumption_co2_intensity', 'emission_factor', 'emissionFactor', 'emission', 'Emission', 'co2', 'CO2']:
    if col in df_emission.columns:
        emission_col = col
        break

if emission_col is None:
    raise ValueError(f"Could not find emission factor column. Available columns: {df_emission.columns.tolist()}")

# Remove timezone if present
if df_emission.index.tz is not None:
    df_emission.index = df_emission.index.tz_localize(None)

# Check if emission factors are in g CO₂/kWh or kg CO₂/kWh
# Convert to g CO₂/kWh if needed (assume > 100 means it's in g, < 1 means it's in kg)
if df_emission[emission_col].max() < 1.0:
    print(f"Converting emission factors from kg CO₂/kWh to g CO₂/kWh")
    df_emission[emission_col] = df_emission[emission_col] * 1000.0
elif df_emission[emission_col].max() > 1000.0:
    print(f"Emission factors appear to be in g CO₂/kWh (max: {df_emission[emission_col].max():.1f} g/kWh)")

# Resample to 15-minute intervals
df_emission_resampled = df_emission[[emission_col]].resample('15min').mean()

print(f"\nAfter processing:")
print(f"  Emission factor range: {df_emission_resampled[emission_col].min():.1f} to {df_emission_resampled[emission_col].max():.1f} g CO₂/kWh")
print(f"  Time range: {df_emission_resampled.index.min()} to {df_emission_resampled.index.max()}")
print(f"\nFirst few rows:")
print(df_emission_resampled.head(30))


## Step 3: Merge All Input Data

**What this step does:** Combines power consumption and emission factor data into a single dataframe. Forward-fills hourly data to fill 15-minute intervals (carries each hour's value forward to 00:15, 00:30, 00:45).

**What you'll see:** Combined data shape, column list, missing values count (should be 0 after forward-fill), and first few rows showing both power and emission factors aligned in time.


In [ ]:
# Merge all data
df = df_power.copy()
df = df.join(df_emission_resampled, how='left')

# Ensure no missing values in critical columns
# Forward-fill power data (hourly data) to fill 15-minute intervals
# This carries the last known power value forward until the next hour's value is available
df['Standortverbrauch'] = df['Standortverbrauch'].ffill().bfill()

# Forward-fill emission factors (hourly data) to fill 15-minute intervals
# This carries the last known emission factor forward until the next hour's value is available
df[emission_col] = df[emission_col].ffill().bfill()

# Rename emission column to standard name
df['Emission Factor (g CO2/kWh)'] = df[emission_col]

print(f"\nCombined data shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nMissing values per column:")
print(df.isna().sum())
print(f"\nFirst few rows:")
print(df[['Standortverbrauch', 'Emission Factor (g CO2/kWh)']].head(30))


## Step 4: Configure Cooling System Properties

**What this step does:** Calculates thermal properties of the cooling system:
- **U-value**: Heat transfer coefficient through walls (W/(m²·K))
- **Overall U**: Total heat transfer coefficient (U × Area) in W/K
- **Heat capacity**: Thermal mass of air and contents (MJ/K)

**Note:** In Version 2, these properties will be updated with fitted values from Step 4.5 (Parameter Calibration) if calibration is successful. Otherwise, initial estimates are used.

**What you'll see:** Calculated U-value, overall heat transfer coefficient, heat capacity, and stored properties dictionaries.


In [ ]:
from utils.insulation_calculator import calculate_heat_transfer_coefficient

# Calculate U-value (initial estimate - will be updated if calibration succeeds)
u_value = calculate_heat_transfer_coefficient(
    insulation_thickness_m=INSULATION_THICKNESS,
    insulation_type=INSULATION_TYPE
)
print(f"Initial U-value estimate: {u_value:.4f} W/(m²·K)")

# Calculate overall heat transfer coefficient (U × Area)
overall_u = u_value * WALL_AREA
print(f"Initial Overall U estimate: {overall_u:.2f} W/K")

# Calculate heat capacity (initial estimate - will be updated if calibration succeeds)
overall_heat_capacity = CONTENT_MASS * SPECIFIC_HEAT_CAPACITY
print(f"Initial Heat capacity estimate: {overall_heat_capacity/1e6:.2f} MJ/K")

# Store properties
mapping_of_walls_properties = {
    SYSTEM_NAME: {
        "area": WALL_AREA,
        "heat_transfer_coef": u_value,
    }
}

mapping_of_content_properties = {
    SYSTEM_NAME: {
        "mass": CONTENT_MASS,
        "specific_heat_capacity": SPECIFIC_HEAT_CAPACITY,
    }
}

print(f"\nWall properties: {mapping_of_walls_properties}")
print(f"Content properties: {mapping_of_content_properties}")
print(f"\nNote: These values will be updated in Step 4.5 if parameter calibration is successful.")


## Step 4.5: Parameter Calibration (NEW in Version 2)

**What this step does:** Fits thermal parameters (Ria, Ci, COP) from measured data using the calibration framework. This provides more accurate system parameters than using estimated values.

**Process:**
1. Loads measured data (processIndoorTemp, processPowerEl, weatherOutdoorTemp)
2. Creates a simulation function that predicts indoor temperature based on power and outdoor temperature
3. Uses the calibration framework to fit Ria (thermal resistance), Ci (thermal capacitance), and COP (coefficient of performance)
4. Converts fitted parameters to U-value and heat capacity for use in subsequent steps

**What you'll see:** 
- Calibration progress and results
- Fitted parameter values (Ria, Ci, COP)
- Converted values (U-value, Heat Capacity)
- Validation plot comparing measured vs. simulated temperatures
- Error metrics (MAE, RMSE)


In [ ]:
# Step 4.5: Parameter Calibration
# This step fits Ria, Ci, and COP from measured data

try:
    from calibration.calibration_framework import calibrate_model, CalibrationResult
    CALIBRATION_AVAILABLE = True
    print("Calibration framework imported successfully")
except ImportError as e:
    print(f"Warning: Could not import calibration framework: {e}")
    print("Skipping parameter calibration - using initial estimates")
    CALIBRATION_AVAILABLE = False

if CALIBRATION_AVAILABLE:
    # Load measured data for calibration
    print(f"\nLoading measured data from: {measured_data_path}")
    if not measured_data_path.exists():
        print(f"Warning: Measured data file not found: {measured_data_path}")
        print("Skipping calibration - using initial estimates")
        CALIBRATION_AVAILABLE = False
    else:
        df_measured = pd.read_csv(measured_data_path, index_col=0, parse_dates=True)
        
        # Remove timezone if present
        if df_measured.index.tz is not None:
            df_measured.index = df_measured.index.tz_localize(None)
        
        # Filter for calibration period
        df_cal = df_measured[
            (df_measured.index >= calibration_start_date) & 
            (df_measured.index <= calibration_end_date)
        ].copy()
        
        # Ensure 15-minute intervals
        df_cal = df_cal.resample('15min').mean()
        
        # Extract required columns
        required_cols = ['processIndoorTemp', 'processPowerEl', 'weatherOutdoorTemp']
        missing_cols = [col for col in required_cols if col not in df_cal.columns]
        
        if missing_cols:
            print(f"Warning: Missing columns in measured data: {missing_cols}")
            print("Skipping calibration - using initial estimates")
            CALIBRATION_AVAILABLE = False
        else:
            # Remove rows with NaN values
            df_cal = df_cal[required_cols].dropna()
            
            if len(df_cal) < 100:
                print(f"Warning: Insufficient data points ({len(df_cal)}) for calibration")
                print("Skipping calibration - using initial estimates")
                CALIBRATION_AVAILABLE = False
            else:
                print(f"Loaded {len(df_cal)} data points for calibration")
                print(f"Time range: {df_cal.index.min()} to {df_cal.index.max()}")
                
                # Extract data arrays
                measured_temp = df_cal['processIndoorTemp'].values
                measured_power = df_cal['processPowerEl'].values  # kW
                outdoor_temp = df_cal['weatherOutdoorTemp'].values
                
                # Get defrost power if available (default to 0)
                defrost_power = df_cal.get('processDefrosting', pd.Series(0.0, index=df_cal.index)).values
                
                # Time step in hours
                dt_hours = 0.25  # 15 minutes
                
                # Create simulation function
                def simulate_cooling_system(params_dict, data):
                    """
                    Simulate cooling system temperature evolution.
                    
                    Uses the thermal model: dT/dt = (outdoor_heat_flow + cooling_heat_flow) / Ci
                    where:
                    - outdoor_heat_flow = (T_outdoor - T_indoor) / Ria
                    - cooling_heat_flow = -(power * COP - defrost) / Ci
                    """
                    Ria = params_dict['Ria']  # K/kW
                    Ci = params_dict['Ci']  # MJ/K
                    COP = params_dict['COP']
                    
                    # Extract inputs
                    power = data['power']  # kW
                    outdoor = data['outdoor_temp']  # °C
                    defrost = data.get('defrost_power', np.zeros_like(power))  # kW
                    initial_temp = data['initial_temp']  # °C
                    
                    # Convert Ci from MJ/K to J/K for calculations
                    Ci_j_per_k = Ci * 1e6
                    
                    # Convert Ria from K/kW to K/W
                    Ria_k_per_w = Ria * 1e-3
                    
                    # Initialize temperature array
                    n = len(power)
                    temp_sim = np.zeros(n)
                    temp_sim[0] = initial_temp
                    
                    # Simulate step by step
                    for i in range(1, n):
                        dt_sec = dt_hours * 3600  # seconds
                        
                        # Current temperature
                        T_current = temp_sim[i-1]
                        
                        # Heat flows
                        # Outdoor heat flow: (T_outdoor - T_indoor) / Ria (in W)
                        outdoor_heat_flow_w = (outdoor[i] - T_current) / Ria_k_per_w
                        
                        # Cooling heat flow: -(power * COP - defrost) (in W)
                        # Note: power * COP gives cooling capacity in kW, convert to W
                        cooling_capacity_w = power[i] * COP * 1000  # W
                        cooling_heat_flow_w = -(cooling_capacity_w - defrost[i] * 1000)
                        
                        # Total heat flow
                        total_heat_flow_w = outdoor_heat_flow_w + cooling_heat_flow_w
                        
                        # Temperature change: dT/dt = Q / C
                        dT_dt = total_heat_flow_w / Ci_j_per_k  # K/s
                        
                        # Update temperature
                        temp_sim[i] = T_current + dT_dt * dt_sec
                    
                    return temp_sim
                
                # Prepare calibration data
                cal_data = {
                    'measured': measured_temp,
                    'power': measured_power,
                    'outdoor_temp': outdoor_temp,
                    'defrost_power': defrost_power,
                    'initial_temp': measured_temp[0],
                }
                
                # Define parameter bounds
                param_definitions = {
                    'Ria': {
                        'value': INITIAL_RIA,
                        'min': 0.5,
                        'max': 10.0,
                    },
                    'Ci': {
                        'value': INITIAL_CI,
                        'min': 0.5,
                        'max': 20.0,
                    },
                    'COP': {
                        'value': INITIAL_COP,
                        'min': 1.0,
                        'max': 8.0,
                    },
                }
                
                print(f"\nStarting calibration...")
                print(f"Initial parameters:")
                print(f"  Ria: {INITIAL_RIA:.3f} K/kW")
                print(f"  Ci: {INITIAL_CI:.3f} MJ/K")
                print(f"  COP: {INITIAL_COP:.3f}")
                
                # Run calibration
                try:
                    result: CalibrationResult = calibrate_model(
                        param_definitions=param_definitions,
                        simulate_func=simulate_cooling_system,
                        data=cal_data,
                        error_metric='mae',
                        method='differential_evolution',
                        maxiter=50,  # Reduced for faster execution
                        popsize=15,
                        tol=1e-3,
                        verbose=True,
                    )
                    
                    # Extract fitted parameters
                    fitted_Ria = result.params['Ria']
                    fitted_Ci = result.params['Ci']
                    fitted_COP = result.params['COP']
                    
                    print(f"\n{'='*70}")
                    print("CALIBRATION COMPLETE")
                    print(f"{'='*70}")
                    print(f"\nFitted Parameters:")
                    print(f"  Ria: {fitted_Ria:.4f} K/kW (initial: {INITIAL_RIA:.4f})")
                    print(f"  Ci: {fitted_Ci:.4f} MJ/K (initial: {INITIAL_CI:.4f})")
                    print(f"  COP: {fitted_COP:.4f} (initial: {INITIAL_COP:.4f})")
                    print(f"\nError Metrics:")
                    print(f"  Initial MAE: {result.initial_error:.4f} °C")
                    print(f"  Final MAE: {result.error:.4f} °C")
                    print(f"  Improvement: {result.improvement:.1f}%")
                    
                    # Convert fitted parameters to U-value and heat capacity
                    # Ria = 1 / (U × Area) in K/kW
                    # Therefore: U × Area = 1 / Ria (in kW/K)
                    # U = (1 / Ria) / Area (in kW/(m²·K)) = 1000 × (1 / Ria) / Area (in W/(m²·K))
                    overall_u_fitted = (1.0 / fitted_Ria) * 1000  # W/K
                    u_value_fitted = overall_u_fitted / WALL_AREA  # W/(m²·K)
                    
                    # Ci is already in MJ/K, convert to J/K for heat capacity
                    overall_heat_capacity_fitted = fitted_Ci * 1e6  # J/K
                    
                    # Calculate equivalent mass and specific heat
                    # Ci = mass × specific_heat_capacity
                    # We'll keep the same specific heat capacity and adjust mass
                    equivalent_mass = overall_heat_capacity_fitted / SPECIFIC_HEAT_CAPACITY  # kg
                    
                    print(f"\nConverted Parameters:")
                    print(f"  U-value: {u_value_fitted:.4f} W/(m²·K)")
                    print(f"  Overall U: {overall_u_fitted:.2f} W/K")
                    print(f"  Heat capacity: {fitted_Ci:.4f} MJ/K")
                    print(f"  Equivalent mass: {equivalent_mass:.1f} kg (at {SPECIFIC_HEAT_CAPACITY} J/(kg·K))")
                    
                    # Update global parameters
                    u_value = u_value_fitted
                    overall_u = overall_u_fitted
                    overall_heat_capacity = overall_heat_capacity_fitted
                    COP = fitted_COP
                    CONTENT_MASS = equivalent_mass
                    
                    # Update property mappings
                    mapping_of_walls_properties = {
                        SYSTEM_NAME: {
                            "area": WALL_AREA,
                            "heat_transfer_coef": u_value,
                        }
                    }
                    
                    mapping_of_content_properties = {
                        SYSTEM_NAME: {
                            "mass": CONTENT_MASS,
                            "specific_heat_capacity": SPECIFIC_HEAT_CAPACITY,
                        }
                    }
                    
                    # Validate fit
                    simulated_temp = simulate_cooling_system(
                        {'Ria': fitted_Ria, 'Ci': fitted_Ci, 'COP': fitted_COP},
                        cal_data
                    )
                    
                    # Calculate additional metrics
                    mae = np.mean(np.abs(measured_temp - simulated_temp))
                    rmse = np.sqrt(np.mean((measured_temp - simulated_temp)**2))
                    r2 = 1 - np.sum((measured_temp - simulated_temp)**2) / np.sum((measured_temp - np.mean(measured_temp))**2)
                    
                    print(f"\nValidation Metrics:")
                    print(f"  MAE: {mae:.4f} °C")
                    print(f"  RMSE: {rmse:.4f} °C")
                    print(f"  R²: {r2:.4f}")
                    
                    # Plot validation
                    fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
                    
                    # Temperature comparison
                    axes[0].plot(df_cal.index, measured_temp, label='Measured', linewidth=2, alpha=0.7)
                    axes[0].plot(df_cal.index, simulated_temp, label='Simulated', linewidth=2, alpha=0.7)
                    axes[0].set_ylabel('Temperature (°C)')
                    axes[0].set_title('Calibration Validation: Measured vs. Simulated Temperature')
                    axes[0].legend()
                    axes[0].grid(True, alpha=0.3)
                    
                    # Residuals
                    residuals = measured_temp - simulated_temp
                    axes[1].plot(df_cal.index, residuals, label='Residuals', linewidth=1, alpha=0.7, color='red')
                    axes[1].axhline(y=0, color='black', linestyle='--', alpha=0.5)
                    axes[1].set_ylabel('Residual (°C)')
                    axes[1].set_xlabel('Time')
                    axes[1].set_title('Residuals (Measured - Simulated)')
                    axes[1].legend()
                    axes[1].grid(True, alpha=0.3)
                    plt.xticks(rotation=45)
                    plt.tight_layout()
                    plt.show()
                    
                    print(f"\n{'='*70}")
                    print("Using fitted parameters for subsequent analysis")
                    print(f"{'='*70}")
                    
                except Exception as e:
                    print(f"Error during calibration: {e}")
                    import traceback
                    traceback.print_exc()
                    print("Using initial estimates")
                    CALIBRATION_AVAILABLE = False

if not CALIBRATION_AVAILABLE:
    print("\nUsing initial parameter estimates (no calibration performed)")
    print(f"  U-value: {u_value:.4f} W/(m²·K)")
    print(f"  Overall U: {overall_u:.2f} W/K")
    print(f"  Heat capacity: {overall_heat_capacity/1e6:.4f} MJ/K")
    print(f"  COP: {COP}")


## Step 5: Create Temperature Schedule (Emission-Based)

**What this step does:** Creates an optimized temperature schedule based on emission factors:
- **Low emissions (clean grid)** → Lower temperature (more cooling when grid is clean)
  - Clean grid = Low CO₂ emissions (g CO₂/kWh) - happens when renewable energy (solar, wind, hydro) is abundant
- **High emissions (dirty grid)** → Higher temperature (less cooling when grid is dirty)
  - Dirty grid = High CO₂ emissions (g CO₂/kWh) - happens when fossil fuels (coal, gas) are used more

**Strategy:** Cool more when electricity is clean (low emissions) to "store" cooling, then use less cooling when electricity is dirty (high emissions). This minimizes total CO₂ emissions.

The schedule respects ramp rate constraints (max temperature change rate) and stays within allowed temperature bounds.

**What you'll see:** Temperature schedule range and mean, first few rows showing schedule vs. emission factors, and a visualization plot showing emission factors over time and the resulting temperature schedule.


In [ ]:
from analysis.emission_schedule_creators import create_emission_like_schedule

# Add temperature constraints to dataframe
df[f'{SYSTEM_NAME}_Default Indoor Temp'] = DEFAULT_INDOOR_TEMP
df[f'{SYSTEM_NAME}_Min Temp Allowed'] = MIN_TEMP_ALLOWED
df[f'{SYSTEM_NAME}_Max Temp Allowed'] = MAX_TEMP_ALLOWED

# Create emission-based temperature schedule
# Lower temperatures when emissions are low (grid is clean), higher when emissions are high (grid is dirty)
df[f'{SYSTEM_NAME}_Temperature Schedule'] = create_emission_like_schedule(
    df=df,
    emission_factor_col='Emission Factor (g CO2/kWh)',
    min_temp_allowed_col=f'{SYSTEM_NAME}_Min Temp Allowed',
    max_temp_allowed_col=f'{SYSTEM_NAME}_Max Temp Allowed',
    ramp_slope_in_k_per_h=abs(COOLING_RAMP_SLOPE_IN_K_PER_H),
    phase_change_temp=PHASE_CHANGE_TEMP if PCM_MASS > 0 else None,
)

print(f"\nTemperature schedule created:")
print(f"  Range: {df[f'{SYSTEM_NAME}_Temperature Schedule'].min():.2f} to {df[f'{SYSTEM_NAME}_Temperature Schedule'].max():.2f} °C")
print(f"  Mean: {df[f'{SYSTEM_NAME}_Temperature Schedule'].mean():.2f} °C")
print(f"\nFirst few rows:")
print(df[[f'{SYSTEM_NAME}_Temperature Schedule', 'Emission Factor (g CO2/kWh)']].head(30))

# Visualize schedule
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(df.index, df['Emission Factor (g CO2/kWh)'], label='Emission Factor', color='red', linewidth=2)
axes[0].set_ylabel('Emission Factor (g CO₂/kWh)')
axes[0].set_title('Emission Factor Over Time')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(df.index, df[f'{SYSTEM_NAME}_Temperature Schedule'], label='Temperature Schedule', color='blue', linewidth=2)
axes[1].axhline(y=DEFAULT_INDOOR_TEMP, color='gray', linestyle='--', label='Default Temp', alpha=0.7)
axes[1].axhline(y=MIN_TEMP_ALLOWED, color='red', linestyle='--', label='Min/Max Allowed', alpha=0.5)
axes[1].axhline(y=MAX_TEMP_ALLOWED, color='red', linestyle='--', alpha=0.5)
axes[1].set_ylabel('Temperature (°C)')
axes[1].set_xlabel('Time')
axes[1].set_title('Emission-Optimized Temperature Schedule')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## Step 6: Calculate Cooling Power (Temperature → Electrical Load)

**What this step does:** Calculates the electrical power consumption required to follow the optimized temperature schedule. This accounts for:
1. **Baseline cooling** (original power consumption)
2. **Additional cooling** needed when temperature is below default (more cooling)
3. **Reduced cooling** when temperature is above default (less cooling)
4. **Temperature change rate** (power needed to change temperature)
5. **System efficiency** (COP - Coefficient of Performance)

**What you'll see:** Power range, mean, and total energy for both baseline and optimized scenarios, plus a comparison plot showing both power curves.


### Understanding the Power Curves

The plot above shows two power curves:

**1. Baseline Power (Blue line - `Standortverbrauch`):**
- **Source:** Measured power consumption from actual EcoCool system operation
- **Temperature:** Constant at default temperature (-20°C)
- **Strategy:** No optimization - follows actual usage patterns
- **Represents:** Real-world power consumption before optimization

**2. Optimized Power (Orange line - `EcoCool_Cooling Power After Optimization`):**
- **Source:** Calculated power consumption based on emission-optimized temperature schedule
- **Temperature:** Varies based on emission factors (lower when grid is clean, higher when dirty)
- **Strategy:** Emission-minimizing - shifts consumption to low-emission times
- **Represents:** Simulated power consumption if system follows the optimized schedule

**How the optimized power is calculated:**
```
Optimized Power = Baseline Power + Additional Cooling Load / (COP × latent_heat_factor)
```

Where additional cooling load accounts for:
- **Temperature deviation:** More cooling needed when schedule temp < default temp
- **Temperature change rate:** Power needed to change temperature (cooling down or warming up)
- **PCM benefits:** Reduced cooling needed when operating near phase change temperature (if PCM present)

**Key insight:** The optimized curve shifts power consumption to times when grid emissions are low, reducing total CO₂ emissions even if total energy consumption is similar.


In [ ]:
from analysis.phase_change_models import calculate_phase_change_cooling_power

# Calculate cooling power based on temperature schedule
df[f'{SYSTEM_NAME}_Cooling Power After Optimization'] = calculate_phase_change_cooling_power(
    df=df,
    cooling_power_col='Standortverbrauch',  # Baseline cooling
    schedule_temp_col=f'{SYSTEM_NAME}_Temperature Schedule',
    dflt_indoor_temp_col=f'{SYSTEM_NAME}_Default Indoor Temp',
    overall_heat_transfer_coef_in_w_per_k=overall_u,
    overall_heat_capacity_in_j_per_k=overall_heat_capacity,
    latent_heat_capacity_in_j_per_kg=LATENT_HEAT_CAPACITY,
    pcm_mass_in_kg=PCM_MASS,
    phase_change_temp_in_c=PHASE_CHANGE_TEMP,
    cop=COP,
    latent_heat_factor=LATENT_HEAT_FACTOR,
)

col = f'{SYSTEM_NAME}_Cooling Power After Optimization'
print(f"\nCooling power after optimization:")
print(f"  Range: {df[col].min():.2f} to {df[col].max():.2f} kW")
print(f"  Mean: {df[col].mean():.2f} kW")
print(f"  Total energy: {df[col].sum() * 0.25:.2f} kWh")
print(f"\nBaseline (before optimization):")
print(f"  Mean: {df['Standortverbrauch'].mean():.2f} kW")
print(f"  Total energy: {df['Standortverbrauch'].sum() * 0.25:.2f} kWh")

# Visualize comparison
plt.figure(figsize=(14, 6))
plt.plot(df.index, df['Standortverbrauch'], label='Baseline', linewidth=2, alpha=0.7)
plt.plot(df.index, df[col], label='After Optimization', linewidth=2)
plt.ylabel('Power (kW)')
plt.xlabel('Time')
plt.title(f'{SYSTEM_NAME} Cooling Power Comparison (Emission-Optimized)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## Step 7: Calculate Grid Power After Optimization

**What this step does:** Calculates grid power consumption before and after optimization. Grid power represents the actual power drawn from the electrical grid (excluding any PV self-consumption if present).

**What you'll see:** Mean grid power before and after optimization, and a comparison plot showing how grid power consumption changes with the optimized schedule.


In [ ]:
# Calculate grid power before (baseline)
df[f'{SYSTEM_NAME}_Grid Power Before'] = df['Standortverbrauch']

# Calculate grid power after optimization
df[f'{SYSTEM_NAME}_Grid Power After'] = df[f'{SYSTEM_NAME}_Cooling Power After Optimization']

print(f"\nGrid Power:")
print(f"  Before (mean): {df[f'{SYSTEM_NAME}_Grid Power Before'].mean():.2f} kW")
print(f"  After (mean): {df[f'{SYSTEM_NAME}_Grid Power After'].mean():.2f} kW")

# Visualize comparison
plt.figure(figsize=(14, 6))
plt.plot(df.index, df[f'{SYSTEM_NAME}_Grid Power Before'], label='Before', linewidth=2, alpha=0.7)
plt.plot(df.index, df[f'{SYSTEM_NAME}_Grid Power After'], label='After', linewidth=2)
plt.ylabel('Power (kW)')
plt.xlabel('Time')
plt.title(f'{SYSTEM_NAME} Grid Power Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## Step 8: Calculate Emissions and Savings

**What this step does:** Calculates CO₂ emissions for both baseline and optimized scenarios by multiplying energy consumption by emission factors. Computes total emissions, emission savings (in kg CO₂), and savings percentage.

**What you'll see:** 
- Total emissions before and after optimization
- Emission savings in kg CO₂ and percentage
- Two plots: hourly emissions comparison and cumulative emissions over time

**Key metric:** Emission savings shows how much CO₂ can be reduced by following the optimized schedule.


In [ ]:
from utils.data_processing import convert_power_to_energy

# Convert power to energy (kWh)
df[f'{SYSTEM_NAME}_Energy Consumption Before (kWh)'] = convert_power_to_energy(
    df[f'{SYSTEM_NAME}_Grid Power Before']
)
df[f'{SYSTEM_NAME}_Energy Consumption After (kWh)'] = convert_power_to_energy(
    df[f'{SYSTEM_NAME}_Grid Power After']
)

# Calculate emissions (g CO₂)
# Emissions = Energy (kWh) × Emission Factor (g CO₂/kWh)
df[f'{SYSTEM_NAME}_Emissions Before (g CO2)'] = (
    df[f'{SYSTEM_NAME}_Energy Consumption Before (kWh)'] * df['Emission Factor (g CO2/kWh)']
)
df[f'{SYSTEM_NAME}_Emissions After (g CO2)'] = (
    df[f'{SYSTEM_NAME}_Energy Consumption After (kWh)'] * df['Emission Factor (g CO2/kWh)']
)

# Calculate hourly emissions for visualization
df[f'{SYSTEM_NAME}_Emissions Before (g CO2/h)'] = (
    df[f'{SYSTEM_NAME}_Grid Power Before'] * df['Emission Factor (g CO2/kWh)']
)
df[f'{SYSTEM_NAME}_Emissions After (g CO2/h)'] = (
    df[f'{SYSTEM_NAME}_Grid Power After'] * df['Emission Factor (g CO2/kWh)']
)

# Calculate total emissions
total_emissions_before = df[f'{SYSTEM_NAME}_Emissions Before (g CO2)'].sum() / 1000  # Convert to kg
total_emissions_after = df[f'{SYSTEM_NAME}_Emissions After (g CO2)'].sum() / 1000  # Convert to kg
emission_savings = total_emissions_before - total_emissions_after
emission_savings_percent = (emission_savings / total_emissions_before * 100) if total_emissions_before > 0 else 0

print(f"\nEmission Analysis:")
print(f"  Total emissions before: {total_emissions_before:.2f} kg CO₂")
print(f"  Total emissions after: {total_emissions_after:.2f} kg CO₂")
print(f"  Emission savings: {emission_savings:.2f} kg CO₂ ({emission_savings_percent:.1f}%)")

# Visualize emissions
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Hourly emissions
axes[0].plot(df.index, df[f'{SYSTEM_NAME}_Emissions Before (g CO2/h)'], 
             label='Before', linewidth=2, alpha=0.7, color='red')
axes[0].plot(df.index, df[f'{SYSTEM_NAME}_Emissions After (g CO2/h)'], 
             label='After', linewidth=2, color='green')
axes[0].set_ylabel('Emissions (g CO₂/h)')
axes[0].set_title('Hourly CO₂ Emissions Comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Cumulative emissions
cumulative_before = df[f'{SYSTEM_NAME}_Emissions Before (g CO2)'].cumsum() / 1000  # kg
cumulative_after = df[f'{SYSTEM_NAME}_Emissions After (g CO2)'].cumsum() / 1000  # kg
axes[1].plot(df.index, cumulative_before, label='Before', linewidth=2, alpha=0.7, color='red')
axes[1].plot(df.index, cumulative_after, label='After', linewidth=2, color='green')
axes[1].set_ylabel('Cumulative Emissions (kg CO₂)')
axes[1].set_xlabel('Time')
axes[1].set_title('Cumulative CO₂ Emissions')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## Step 9: Validate Temperature Schedule

**What this step does:** Validates that the temperature schedule:
1. Stays within allowed temperature bounds (min/max)
2. Respects ramp rate constraints (max temperature change per time step)

**What you'll see:** Validation results showing whether all temperatures are within bounds, min/max values in the schedule, and ramp rate compliance (number of violations if any).


In [ ]:
# Simulate temperature based on cooling power
# This is a simplified validation - in practice, you would compare with measured temperatures
df[f'{SYSTEM_NAME}_Simulated Temperature'] = df[f'{SYSTEM_NAME}_Temperature Schedule']

# Check if schedule stays within bounds
within_bounds = (
    (df[f'{SYSTEM_NAME}_Temperature Schedule'] >= MIN_TEMP_ALLOWED) &
    (df[f'{SYSTEM_NAME}_Temperature Schedule'] <= MAX_TEMP_ALLOWED)
).all()

print(f"\nTemperature Schedule Validation:")
print(f"  All temperatures within bounds: {within_bounds}")
print(f"  Min in schedule: {df[f'{SYSTEM_NAME}_Temperature Schedule'].min():.2f} °C (allowed: {MIN_TEMP_ALLOWED} °C)")
print(f"  Max in schedule: {df[f'{SYSTEM_NAME}_Temperature Schedule'].max():.2f} °C (allowed: {MAX_TEMP_ALLOWED} °C)")

# Check ramp rate compliance
temp_changes = df[f'{SYSTEM_NAME}_Temperature Schedule'].diff().abs()
time_step_hours = 0.25  # 15 minutes
max_allowed_change = abs(COOLING_RAMP_SLOPE_IN_K_PER_H) * time_step_hours
ramp_violations = (temp_changes > max_allowed_change).sum()

print(f"\nRamp Rate Validation:")
print(f"  Max allowed change per 15min: {max_allowed_change:.3f} °C")
print(f"  Max actual change: {temp_changes.max():.3f} °C")
print(f"  Ramp violations: {ramp_violations}")


## Step 10: Generate Reports

**What this step does:** Exports results to Excel files:
- `results.xlsx`: Complete time series data with all calculated columns
- `summary.xlsx`: Summary statistics including total energy, emissions, and savings

**What you'll see:** File paths where reports were saved and a summary table showing key metrics (energy consumption, emissions, savings).


In [ ]:
# Prepare DataFrame for export
system_df = df[[
    'Standortverbrauch',
    'Emission Factor (g CO2/kWh)',
    f'{SYSTEM_NAME}_Temperature Schedule',
    f'{SYSTEM_NAME}_Cooling Power After Optimization',
    f'{SYSTEM_NAME}_Grid Power Before',
    f'{SYSTEM_NAME}_Grid Power After',
    f'{SYSTEM_NAME}_Energy Consumption Before (kWh)',
    f'{SYSTEM_NAME}_Energy Consumption After (kWh)',
    f'{SYSTEM_NAME}_Emissions Before (g CO2/h)',
    f'{SYSTEM_NAME}_Emissions After (g CO2/h)',
    f'{SYSTEM_NAME}_Simulated Temperature',
]].copy()

# Rename columns (remove system name prefix)
rename_dict = {
    f'{SYSTEM_NAME}_Temperature Schedule': 'Temperature Schedule',
    f'{SYSTEM_NAME}_Cooling Power After Optimization': 'Cooling Power After Optimization',
    f'{SYSTEM_NAME}_Grid Power Before': 'Grid Power Before',
    f'{SYSTEM_NAME}_Grid Power After': 'Grid Power After',
    f'{SYSTEM_NAME}_Energy Consumption Before (kWh)': 'Energy Consumption Before (kWh)',
    f'{SYSTEM_NAME}_Energy Consumption After (kWh)': 'Energy Consumption After (kWh)',
    f'{SYSTEM_NAME}_Emissions Before (g CO2/h)': 'Emissions Before (g CO2/h)',
    f'{SYSTEM_NAME}_Emissions After (g CO2/h)': 'Emissions After (g CO2/h)',
    f'{SYSTEM_NAME}_Simulated Temperature': 'Simulated Temperature',
}
system_df = system_df.rename(columns=rename_dict)

# Save Excel report
excel_path = os.path.join(report_directory, 'results.xlsx')
system_df.to_excel(excel_path, index=True)
print(f"\nExcel report saved: {excel_path}")

# Create summary report
summary = {
    'Metric': [
        'Total Energy Before (kWh)',
        'Total Energy After (kWh)',
        'Energy Savings (kWh)',
        'Total Emissions Before (kg CO₂)',
        'Total Emissions After (kg CO₂)',
        'Emission Savings (kg CO₂)',
        'Emission Savings (%)',
    ],
    'Value': [
        df[f'{SYSTEM_NAME}_Energy Consumption Before (kWh)'].sum(),
        df[f'{SYSTEM_NAME}_Energy Consumption After (kWh)'].sum(),
        df[f'{SYSTEM_NAME}_Energy Consumption Before (kWh)'].sum() - df[f'{SYSTEM_NAME}_Energy Consumption After (kWh)'].sum(),
        total_emissions_before,
        total_emissions_after,
        emission_savings,
        emission_savings_percent,
    ]
}
summary_df = pd.DataFrame(summary)

summary_path = os.path.join(report_directory, 'summary.xlsx')
summary_df.to_excel(summary_path, index=False)
print(f"Summary report saved: {summary_path}")
print(f"\nSummary:")
print(summary_df.to_string(index=False))


## Summary


In [ ]:
print("="*70)
print("MANUAL ANALYSIS COMPLETE")
print("="*70)
print(f"\nResults saved to: {report_directory}")
print(f"\nSystem: {SYSTEM_NAME}")
print(f"\nKey Results:")
print(f"  Emission savings: {emission_savings:.2f} kg CO₂ ({emission_savings_percent:.1f}%)")
print(f"  Energy before: {df[f'{SYSTEM_NAME}_Energy Consumption Before (kWh)'].sum():.2f} kWh")
print(f"  Energy after: {df[f'{SYSTEM_NAME}_Energy Consumption After (kWh)'].sum():.2f} kWh")

if os.path.exists(report_directory):
    files = os.listdir(report_directory)
    print(f"\nGenerated files:")
    for file in files:
        file_path = os.path.join(report_directory, file)
        size = os.path.getsize(file_path) / 1024  # KB
        print(f"  - {file}: {size:.1f} KB")
